# Module 11: Loss, Optimization & Training Loop

## Learning Objectives
- Implement cross-entropy loss for language modeling
- Build a training loop with optimizer steps
- Implement learning rate schedules and gradient clipping
- Connect to Kyro's ModelLoader

## Concept: Training a Causal LM

The training loop consists of:
1. Forward pass → compute logits
2. Cross-entropy loss → measure prediction error
3. Backward pass → compute gradients
4. Optimizer step → update weights

**Kyro mapping**: Trained weights map to `src/model/loader.rs`

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class TinyLM(nn.Module):
    def __init__(self, vocab_size=50, d_model=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
    def forward(self, x):
        emb = self.embed(x)
        return self.lm_head(emb)

model = TinyLM(vocab_size=50, d_model=16)
optimizer = optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

num_epochs = 5
for epoch in range(num_epochs):
    total_loss = 0
    for step in range(10):
        input_ids = torch.randint(0, 50, (4, 8))
        target_ids = torch.randint(0, 50, (4, 8))
        logits = model(input_ids)
        loss = criterion(logits.view(-1, 50), target_ids.view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / 10
    print(f'Epoch {epoch+1}: avg_loss={avg_loss:.4f}')

print('\nTraining complete!')
print('Trained weights map to safetensors for Kyro ModelLoader')

## Step 1: Learning Rate Schedule

Adjust the learning rate during training for better convergence.

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

model = TinyLM(vocab_size=50, d_model=16)
optimizer = optim.AdamW(model.parameters(), lr=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=50)

print('Cosine annealing learning rate schedule:')
for epoch in range(10):
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    print(f'  Epoch {epoch}: lr={lr:.6f}')

## Exercises
1. Implement checkpoint saving/resuming.
2. Export model weights and load into Kyro.
3. Try different optimizers (Adam vs SGD vs AdamW).